# 00: 環境初期化とサンプルデータ生成（Usagi Craft & Cafe）

## このノートブックで学習できること

- クラウドの2バケット構成（連携用 Landing バケットとレイクハウス本体バケット）をローカルディレクトリ `./storage/data-landing/` と `./storage/lakehouse-main/` として擬似再現する方法
- 仮想企業「Usagi Craft & Cafe」の4つのソースデータ（`crm_customers`、`pos_transactions`、`erp_products`、`store_master`）を Day 1〜Day 3 の3タイムスライスで生成する手順
- Polars と DuckDB を使って、Day 2 のスキーマ変化（Schema Drift）や型異常、Day 3 の異常値混入を事前に確認する方法

## 到達ゴール

- `./storage/data-landing/day1` 〜 `day3` に4種類のソースデータが出力され、以降のノートブック（01〜06）を実行できる状態を整える
- Day 1（初期状態）、Day 2（差分・列追加・型異常・列名変更・退会）、Day 3（売上異常値20%混入）のデータ変化を把握できる

## なぜこの構成が必要か

従来のデータ基盤では、上流システムから出力される生データと加工済みの分析テーブルを同じ領域に混在させたり、全件ダンプをそのまま本番テーブルへ上書きして履歴や監査性を失うケースが多く見られました。連携用領域（Landing / DMZ）とレイクハウス本体（Bronze / Silver / Gold / PII Vault）を分離することで、上流の障害やスキーマ変化を安全に受け止められる構成を作ります。


## 1. ストレージディレクトリの初期化

このセルでは、`src/catalog_helper.py` の `init_storage(clean=True)` を呼び出し、`./storage/data-landing/` と `./storage/lakehouse-main/`（`bronze/`, `silver/`, `gold/`, `pii_vault/`）を初期化します。既存の検証データがある場合は一度削除して作り直します。


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.catalog_helper import (
    LANDING_DIR,
    LAKEHOUSE_DIR,
    get_duckdb_conn,
    get_iceberg_catalog,
    init_storage,
    register_iceberg_table,
)

paths = init_storage(clean=True)
for k, v in paths.items():
    print(f"{k}: {v.relative_to(ROOT_DIR) if v != ROOT_DIR else '.'}")


`landing: storage/data-landing` と `lakehouse: storage/lakehouse-main` が初期化されました。

次のセルでは、`src/data_generator.py` の `generate_all_days` を実行し、Day 1〜Day 3 の生データを生成してファイル別の件数を表示します。


In [ ]:
import polars as pl
from src.data_generator import generate_all_days

summary = generate_all_days(LANDING_DIR, seed=42)
rows = [
    {"day": day, "file": fname, "rows": cnt}
    for day, files in summary.items()
    for fname, cnt in files.items()
]
pl.DataFrame(rows)


Day 1〜Day 3 の各ファイルが生成されました。`crm_customers.csv` は 500 → 515 → 525 件と日次全件スナップショットで増えており、`pos_transactions.jsonl` は各日 300 件の差分データになっています。

次のセルでは、Polars で Day 1 と Day 2 の `pos_transactions.jsonl` を読み込み、スキーマの違いと型異常（`quantity = "two"` など）を確認します。


In [ ]:
pos_d1 = pl.read_ndjson(LANDING_DIR / "day1" / "pos_transactions.jsonl")
pos_d2 = pl.read_ndjson(LANDING_DIR / "day2" / "pos_transactions.jsonl", infer_schema_length=500)

print("Day 1 columns:", pos_d1.columns)
print("Day 2 columns:", pos_d2.columns)
print("Day 2 quantity dtype:", pos_d2.schema["quantity"])
pos_d2.filter(~pl.col("quantity").str.contains(r"^[0-9]+$")).head(5)


Day 2 では `payment_method` と `discount_amount` の2列が追加され、さらに `quantity` に文字列（`"two"`, `"three"`, `"N/A"`）が混入しているため列全体が文字列型として推論されていることが確認できました。

次のセルでは、DuckDB を使って Day 3 の `pos_transactions.jsonl` に含まれる `total_amount` の異常値（マイナス値および NULL）の件数を集計します。


In [ ]:
con = get_duckdb_conn()
d3_path = str(LANDING_DIR / "day3" / "pos_transactions.jsonl")
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE total_amount < 0) AS negative_rows,
    COUNT(*) FILTER (WHERE total_amount IS NULL) AS null_rows
FROM read_json_auto('{d3_path}')
""").pl()


Day 3 の 300 件のうち、マイナス値が 30 件、NULL が 30 件（合計 60 件 = 20%）含まれていることが確認できました。これらの異常データは後のノートブック（03 および 05）で安全に隔離・検査します。
